# Εκπαίδευση TabDDPM στο Google Colab

**Πριν ξεκινήσεις:** `Runtime → Change runtime type → T4 GPU`.

Στο Google Drive χρειάζεται ο φάκελος `MyDrive/thesis/model_input/` με:
- τα 9 αρχεία του `data/model_input/` (`A_train.parquet` … `D_val.parquet`, `london_test.parquet`)
- το `artifacts/model_input/schema.json`

Τα αποτελέσματα γράφονται στο `MyDrive/thesis/outputs/`, ώστε να μη χάνονται όταν κλείσει η συνεδρία.

## 1. Έλεγχος GPU

In [ ]:
!nvidia-smi

## 2. Εγκατάσταση (μία φορά ανά συνεδρία, ~3–5 λεπτά)

Το Colab τρέχει πλέον Python 3.13, για την οποία δεν υπάρχουν τα torch 2.2.2 και numpy 1.26.4 που απαιτεί το synthcity. Γι' αυτό φτιάχνεται ένα ξεχωριστό περιβάλλον **Python 3.11** με το `uv`, με τις ίδιες εκδόσεις όπως το `requirements.txt`. Τα scripts τρέχουν με αυτό (`/content/py311/bin/python`). **Δεν χρειάζεται Restart session.**

In [ ]:
!pip install -q uv
!uv venv /content/py311 --python 3.11 -q
!uv pip install -q --python /content/py311/bin/python torch==2.2.2 numpy==1.26.4 pandas==2.3.3 pyarrow==24.0.0 scipy==1.17.1 scikit-learn==1.8.0 xgboost==2.1.4 synthcity==0.2.12 opacus==1.4.0

## 3. Drive, κώδικας και φάκελοι

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = '/content/thesis-bikeshare-synthesis'
DRIVE = '/content/drive/MyDrive/thesis'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/LefieTsimplekas/thesis-bikeshare-synthesis', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
os.chdir(REPO)

# Είσοδοι από το Drive
os.makedirs('data', exist_ok=True); os.makedirs('artifacts/model_input', exist_ok=True)
if not os.path.exists('data/model_input'):
    os.symlink(f'{DRIVE}/model_input', 'data/model_input')
if not os.path.exists('artifacts/model_input/schema.json'):
    os.symlink(f'{DRIVE}/model_input/schema.json', 'artifacts/model_input/schema.json')

# Έξοδοι στο Drive
for local, remote in [('data/synthetic', 'synthetic'), ('models', 'models'), ('artifacts/training', 'training')]:
    os.makedirs(f'{DRIVE}/outputs/{remote}', exist_ok=True)
    if not os.path.exists(local):
        os.symlink(f'{DRIVE}/outputs/{remote}', local)

print(sorted(os.listdir('data/model_input')))

PY = '/content/py311/bin/python'
subprocess.run([PY, '-c', "import sys, torch, numpy; print('python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| numpy', numpy.__version__)"], check=True)

## 4. Πιλοτική εκτέλεση (σενάριο D, 50.000 γραμμές, 5.000 βήματα)

Ελέγχει ότι όλα δουλεύουν και μετρά την ταχύτητα. Στο «Quick check» στο τέλος, τα ποσοστά και οι διάμεσοι του συνθετικού πρέπει να είναι κοντά σε αυτούς του train.

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D --pilot-rows 50000 --steps 5000 --gen-rows 20000 2>&1 | grep -v 'Epoch'

## 5. Πλήρης εκπαίδευση (ένα σενάριο ανά κελί)

Τρέξε τα κελιά ένα ένα. Αν η συνεδρία κλείσει, ό,τι έχει τελειώσει είναι ήδη στο Drive· ξανατρέχεις τα κελιά 2 και 3 και μετά το επόμενο σενάριο.

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario D 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario A 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario B 2>&1 | grep -v 'Epoch'

In [ ]:
!/content/py311/bin/python src/07_train_tabddpm.py --scenario C 2>&1 | grep -v 'Epoch'